In [19]:
from dotenv import load_dotenv
load_dotenv()

True

In [20]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [21]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str
    app_name: str

In [22]:
from langchain.tools import tool, ToolRuntime
from typing import Annotated
from langchain_core.tools import InjectedToolArg

@tool
def get_user_info(runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    현재 사용자의 정보 조회 (시스템 내부용 도구)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name

    memories = runtime.store.search((user_id, app))

    if not memories:
        return "기록된 정보 없음"
    
    results = []
    for item in memories:
        data = item.value

        if "personal_info" in data:
            results.append(f"- 개인정보: {data["personal_info"]}")
        if "preferece" in data:
            results.append(f"- 선호도: {data["preferece"]}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"


In [23]:
from typing import TypedDict

class UserInfo(TypedDict):
    personal_info: str
    preferece: str


In [24]:
import uuid

@tool
def save_user_info(user_info: UserInfo, runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    사용자의 정보를 저장하거나 업데이트
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    memory_key = str(uuid.uuid4())
    store.put((user_id, app), memory_key, user_info)

    return f"정보가 안전하게 저장되었습니다. (ID: {memory_key})"

In [25]:
system_message = "당신은 사용자의 정보를 기억하는 비서입니다. 사용자가 자신의 정보를 말하면 반드시 'save_user_info' 도구를 사용하여 저장하세요."

In [26]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[get_user_info, save_user_info],
    context_schema=Context,
    store=store,
    system_prompt=system_message
)

In [27]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 이제 '김일남'이야"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

d:\wsh\langchain-workspace\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


In [28]:
response

{'messages': [HumanMessage(content="내 이름은 이제 '김일남'이야", additional_kwargs={}, response_metadata={}, id='adf000b0-df4b-43ee-888c-0559fba5ba2e'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"preferece": "\\uc55e\\uc73c\\ub85c \'\\uae40\\uc77c\\ub0a8\'\\uc774\\ub77c\\ub294 \\uc774\\ub984\\uc73c\\ub85c \\ubd88\\ub9ac\\uae30\\ub97c \\uc6d0\\ud558\\uc2ec", "personal_info": "\\uc131\\ud568: \\uae40\\uc77c\\ub0a8"}}'}, '__gemini_function_call_thought_signatures__': {'e5e9295e-242d-4514-bfd2-7b5b6ae577e6': 'EuoHCucHAb4+9vub6LPLHvnANF2sJ35OKv/D0A5HGb+AKXah2h87aCyG5NAZyMKeOJzRD6AYr7wtT9A2f1CBwPyZVXma6hjVYJoUjLi2srnmpRxtVX+Yh51xSXuFZ5Z0uZDDruYPKE+yKdVb87Ycms70BeG/IO2leAQ9dDQkGIlEGrhrihLOo+YHifhOx9FlREEIZt9FWDiZ1KNl3gbhx0285LHeNrYP9YrR0gbuLAMZtgoRxb0ZO17Dr9wSeG4k1Bk4X5hgHvDvnMMEUkRQ5wC+A2Zel+r8FiGNJ7Ru7XCWJTeGF8qK0z0MPJI8cYEzIYy9rvmuhMRb6wBivuoQAyJSjjfHIZvac7679oc3lYT+2wjViMSoBK0txuspH2tusXJtYMsA457RyBui4vvpsdhkFsYyNJ9xfpcobbMNH

In [29]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 아아를 좋아해"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response

d:\wsh\langchain-workspace\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='나는 아아를 좋아해', additional_kwargs={}, response_metadata={}, id='58710a42-447a-4359-8df2-65f8f42ffc11'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"preferece": "\\uc544\\uc774\\uc2a4 \\uc544\\uba54\\ub9ac\\uce74\\ub178(\\uc544\\uc544)\\ub97c \\uc88b\\uc544\\ud568", "personal_info": "\\uc54c\\ub824\\uc9c4 \\uc815\\ubcf4 \\uc5c6\\uc74c"}}'}, '__gemini_function_call_thought_signatures__': {'ece32abb-1739-4d46-a448-7c7946978177': 'Er0JCroJAb4+9vs2em6jUJgKGUJaAPhI+i6Db7LI/eZPLMvG/r4LqDuL0FDwLmflxYXXDdV8j6Q0sFQ6wKiuxWkfkcc0+0AeTEmpL1fFcRgtWLu4GJqnL2EQ0bdcidmAui2aOFNRMlC8XTubK7lY3FLWn2+RH2pba8pG5ovJ60Bp5Al4cVGiECsBClJnuUyLwUhrXbSCaklOkh4lRAm9VZu6f5W20PQFtZTwRz0YPqaLAh1bRQ8EUH0cfy/eWJhjLtfaZ9AmS8ScRGt0agld3XaA9hQOHZFfne/JiejT5Wy9zudV65w+/PBQ+6hpAkzYlBKUAgJwyuJr1Ah39UFVQtnWN7JNuAnXUcEBRmH8mn1sgNooB4ah+4qgvCOE8k2qnL0pxzXR1qYPNqOGMjQn3kk99Zm0tdKWVYS63ir70RFnXqrAKUFHs5qw/6+J1VqQLwJIU1TG+S+qBry

In [30]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 랭체인을 공부하고 있어. 학습용 챗봇 프로젝트를 진행하고 있는 중이야."}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response

d:\wsh\langchain-workspace\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='나는 랭체인을 공부하고 있어. 학습용 챗봇 프로젝트를 진행하고 있는 중이야.', additional_kwargs={}, response_metadata={}, id='32f17de4-e4e9-4b6e-baba-90d5b456af5f'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"preferece": "\\ud559\\uc2b5\\uc6a9 \\ucc57\\ubd07 \\ud504\\ub85c\\uc81d\\ud2b8\\ub97c \\uc9c4\\ud589 \\uc911\\uc784", "personal_info": "\\ub7ad\\uccb4\\uc778(LangChain)\\uc744 \\uacf5\\ubd80\\ud558\\uace0 \\uc788\\uc74c"}}'}, '__gemini_function_call_thought_signatures__': {'18a9b993-0a0f-4406-86c7-cf0f44149226': 'Et8ECtwEAb4+9vt94KEfsLDB2m7373lf5OPHWNiZ7h5c16HX06fh2ecP9s47bp8mTAsnFH5qtYQyQV1bZSr0qlxvPm3D6BdsAXvfejId3vWU8cPcBZOlQpm5p/oNDBCo7K+j1yn4ooqahxiGwFYSpEKbV0l7E4J3NpK6H3K38B6UwBAxNEzJYdSK21PeMV2DBMrUXkiC5Uaef16vTrdq+ENv6walyxaA8457viGA+RNm5jEeOY/A6Yr/5TXpgvPfsne/Ct3R5oxtJLlWuS+A3rzWI0hXUR3Y3c2usGAFZ4KTL8Eur/9bOWxwPPDkpsG1Q8KEszbbXtXCDkCQBUY7rw/GkD7nZglKJDzqZ/8NGVSIwybhH6//Gw2YVUJBvujfAlZP7/UNTAn1FBQ

In [31]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 뭐고, 내가 좋아하는 것은 뭐야?"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
response

d:\wsh\langchain-workspace\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='내 이름은 뭐고, 내가 좋아하는 것은 뭐야?', additional_kwargs={}, response_metadata={}, id='5955594e-647f-411d-8c4f-99114d0f06b2'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_user_info', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'03b9dc72-2936-4b76-b2d3-0e07222b540c': 'Ep8ECpwEAb4+9vsdwF7TsUU3pT90CA3B2y5MsmXpqe/Je64XdwsvZXr/+naTvz2OfR/E9DqusoelX5uAwWnl94729Ft+uIYhQs/S1ioJDcXrxcs1Q/7VCXTjDOjHREGfRqojyz51NQcuCVb13J78QJmjq3FTqPTIDTZB+R3zoTq3xI7xcMZNh5v3F2CKJloP0eWEWHx6aMYsjQNI6hINA8c1MeqgygGiQsFqtovxiLbrjnbisKu1FKFXFaXbNY/9vutw/6QrDDNDuEqVEXm63dILI2EUj+Y4W1N3FrAoJutW3WI2cz683bSEbtm4RIDW7lXlRC/yMvs2QsC2hDk8UbDPkTK1D/OBFKLOrRuaVqAJmabnnaNxJz0PGHJdE7UZubLaKaEavNGLaM2Pv28jnEb2dMzPndI3fazzIIqXIs9SO6mzbevCu6fDeJcxlZBFdC/7UUEV2pIvLvv450QCc4/G7nJ5Eri9epPdxPRi+EsrzZiAfBya5LgWAfseYe25vYg5BakCxKMTb4jgtumuHfa7+xu42nsLWJYl+0ZfbUkY5sJosiLvZO+bROfAKA5RZKKfe9/rtulm170tzcWuB/dkPKtz8gX6QfAIrHFeYQBqqyyZQYz70IdV577JXCIkzRaezhUnyIsy8m